# 04 - LLM Integration (Groq / Llama 3.3 70B)
Generate Vietnamese analysis for all 3 forecasting questions + executive summary.

**Yêu cầu:** Set `GROQ_API_KEY` trong file `.env` trước khi chạy.

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv('../.env', override=True)

DATA = Path('../data/processed')
FCAST = DATA / 'forecasts'

GROQ_API_KEY = os.getenv('GROQ_API_KEY', '')
has_key = bool(GROQ_API_KEY) and GROQ_API_KEY != 'your_groq_api_key_here'
print(f'Groq API key configured: {has_key}')
print(f'Key preview: {GROQ_API_KEY[:10]}...' if has_key else 'No key found')

if has_key:
    from groq import Groq
    client = Groq(api_key=GROQ_API_KEY)
    MODEL = 'llama-3.3-70b-versatile'
    print(f'Using model: {MODEL}')
else:
    MODEL = 'not_configured'

Groq API key configured: True
Key preview: gsk_hr2Dme...


Using model: llama-3.3-70b-versatile


In [2]:
monthly = pd.read_parquet(FCAST / 'q1_monthly_forecast.parquet')
group_fc = pd.read_parquet(FCAST / 'q1_group_forecast.parquet')
top20 = pd.read_parquet(FCAST / 'q1_top20_skus.parquet')
comparison = pd.read_parquet(FCAST / 'q1_model_comparison.parquet')
color_fc = pd.read_parquet(FCAST / 'q2_color_forecast.parquet')
slow_moving = pd.read_parquet(FCAST / 'q2_slow_moving.parquet')
dealers = pd.read_parquet(FCAST / 'q3_dealer_probability.parquet')
churn = pd.read_parquet(FCAST / 'q3_churn_risk.parquet')
print('All forecast data loaded.')

All forecast data loaded.


In [3]:
def ask_llm(prompt, system_msg=None):
    if not has_key:
        return f'[LLM chưa được cấu hình - cần GROQ_API_KEY]\n\nPrompt gửi đi:\n{prompt[:500]}...'
    messages = []
    if system_msg:
        messages.append({'role': 'system', 'content': system_msg})
    messages.append({'role': 'user', 'content': prompt})
    resp = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.3, max_tokens=2000)
    return resp.choices[0].message.content

SYSTEM = 'Bạn là chuyên gia phân tích dữ liệu cho Thống Nhất Bike - nhà sản xuất xe đạp hàng đầu Việt Nam. Mô hình kinh doanh B2B: bán qua đại lý, không bán trực tiếp cho người dùng cuối. Hãy phân tích dữ liệu và đưa ra nhận định chiến lược bằng tiếng Việt, ngắn gọn và thực tiễn.'
print('LLM helper ready.')

LLM helper ready.


In [4]:
q1_data = f"""DỰ BÁO DOANH SỐ Q2/2026:\n\nDoanh thu theo tháng:\n{monthly[['month_name','revenue','quantity']].to_string(index=False)}\n\nDoanh thu theo nhóm sản phẩm:\n{group_fc.to_string(index=False)}\n\nTop 10 SKU dự kiến bán chạy nhất:\n{top20[['rank','product_name','group_code','pred_quantity','pred_revenue']].head(10).to_string(index=False)}\n\nSo sánh model:\n{comparison.to_string(index=False)}"""
q1_prompt = f"""Dựa trên dữ liệu dự báo doanh số Q2/2026 của Thống Nhất Bike dưới đây:\n\n{q1_data}\n\nHãy:\n1. Phân tích xu hướng doanh thu Q2/2026\n2. Nhận định nhóm sản phẩm nào sẽ dẫn đầu và tại sao\n3. Đánh giá Top 10 SKU bán chạy\n4. Đề xuất 3-5 chiến lược kinh doanh cụ thể cho Q2/2026"""
q1_analysis = ask_llm(q1_prompt, SYSTEM)
print('=== Q1 Analysis ===')
print(q1_analysis)

=== Q1 Analysis ===
Dựa trên dữ liệu dự báo doanh số Q2/2026 của Thống Nhất Bike, dưới đây là phân tích và nhận định chiến lược:

**1. Phân tích xu hướng doanh thu Q2/2026:**
Doanh thu dự kiến tăng dần qua các tháng, từ 8.132 tỷ đồng vào tháng 4 lên 10.160 tỷ đồng vào tháng 6. Điều này cho thấy xu hướng tăng trưởng tích cực trong quý 2.

**2. Nhận định nhóm sản phẩm nào sẽ dẫn đầu và tại sao:**
Nhóm sản phẩm CITYBIKE_P dự kiến dẫn đầu với doanh thu tăng dần qua các tháng (3.805 tỷ đồng vào tháng 4, 3.819 tỷ đồng vào tháng 5 và 4.774 tỷ đồng vào tháng 6). Điều này có thể do nhu cầu về xe đạp thành phố tăng cao trong mùa hè, và CITYBIKE_P là dòng sản phẩm phù hợp với nhu cầu này.

**3. Đánh giá Top 10 SKU bán chạy:**
Top 10 SKU bán chạy dự kiến chủ yếu là các sản phẩm thuộc nhóm CITYBIKE_P và KIDBIKE_1. Điều này cho thấy nhu cầu về xe đạp thành phố và xe đạp trẻ em tăng cao. Các sản phẩm như Xe đạp Thống Nhất Puppy 20 Hồng và Xe đạp Thống Nhất LD 26 Kem dự kiến bán chạy nhất.

**4. Đề xu

In [5]:
slow_count = slow_moving['status'].value_counts().to_dict()
top_colors = color_fc.groupby('color_std')['predicted_qty_share'].mean().nlargest(10)
q2_data = f"""DỰ BÁO MÀU SẮC Q2/2026:\n\nTop 10 màu phổ biến nhất:\n{top_colors.to_string()}\n\nSKU phân loại: {slow_count}\n\nDanh sách SKU bán chậm:\n{slow_moving[slow_moving['status']=='Bán chậm'][['product_name','color_std','group_code','velocity_ratio','status']].head(10).to_string(index=False)}"""
q2_prompt = f"""Dựa trên dữ liệu dự báo màu sắc Q2/2026 của Thống Nhất Bike:\n\n{q2_data}\n\nHãy:\n1. Phân tích xu hướng màu sắc theo mùa (Q2 là mùa hè)\n2. Nhận định tỷ trọng cơ cấu màu sắc dự kiến\n3. Đánh giá danh sách SKU bán chậm và đề xuất xử lý\n4. Đề xuất chiến lược màu sắc cho sản phẩm mới"""
q2_analysis = ask_llm(q2_prompt, SYSTEM)
print('=== Q2 Analysis ===')
print(q2_analysis)

=== Q2 Analysis ===
Dựa trên dữ liệu dự báo màu sắc Q2/2026 của Thống Nhất Bike, dưới đây là phân tích và đề xuất chiến lược:

1. **Phân tích xu hướng màu sắc theo mùa**: Q2 là mùa hè, và dữ liệu dự báo cho thấy các màu sắc phổ biến nhất là ghi, đen, kem, hồng, xanh. Điều này cho thấy người tiêu dùng có xu hướng chọn màu sắc tươi sáng, mát mẻ và trẻ trung trong mùa hè. Màu ghi và đen vẫn là lựa chọn phổ biến do tính thời trang và dễ kết hợp.

2. **Nhận định tỷ trọng cơ cấu màu sắc dự kiến**: Dựa trên dữ liệu, tỷ trọng cơ cấu màu sắc dự kiến sẽ tập trung vào các màu sắc sau:
 * Màu ghi, đen, kem: 60% (doanh số bán hàng cao)
 * Màu hồng, xanh: 20% (doanh số bán hàng trung bình)
 * Các màu sắc khác: 20% (doanh số bán hàng thấp)

3. **Đánh giá danh sách SKU bán chậm và đề xuất xử lý**: Danh sách SKU bán chậm cho thấy các sản phẩm có màu sắc không phổ biến hoặc không phù hợp với thị trường. Đề xuất xử lý:
 * Xe đạp Thống Nhất Neo 20-02 Đỏ tươi, Coban, Hồng: xem xét thay đổi màu sắc hoặc thi

In [6]:
priority_dist = dealers['marketing_priority'].value_counts().to_dict()
segment_dist = dealers['rfm_segment'].value_counts().to_dict()
q3_data = f"""DỰ BÁO HOẠT ĐỘNG ĐẠI LÝ:\n\nPhân khúc tiếp thị: {priority_dist}\nPhân khúc RFM: {segment_dist}\n\nTổng đại lý: {len(dealers)}\nĐại lý xác suất đặt hàng > 70%: {(dealers['prob_order_30d'] > 0.7).sum()}\nĐại lý xác suất đặt hàng < 30%: {(dealers['prob_order_30d'] < 0.3).sum()}\n\nTop 10 đại lý nguy cơ cao nhất:\n{churn[['customer_code','recency','total_orders','prob_order_30d','rfm_segment','marketing_priority','risk_level']].head(10).to_string(index=False)}"""
q3_prompt = f"""Dựa trên dữ liệu dự báo hoạt động đại lý của Thống Nhất Bike:\n\n{q3_data}\n\nHãy:\n1. Phân tích tình hình hoạt động đại lý tổng thể\n2. Nhận định về nhóm đại lý nguy cơ rời bỏ\n3. Đề xuất chiến lược giữ chân đại lý theo từng phân khúc\n4. Đề xuất mức độ ưu tiên tiếp thị cho từng nhóm"""
q3_analysis = ask_llm(q3_prompt, SYSTEM)
print('=== Q3 Analysis ===')
print(q3_analysis)

=== Q3 Analysis ===
**1. Phân tích tình hình hoạt động đại lý tổng thể**

Tổng đại lý của Thống Nhất Bike là 678. Phân khúc tiếp thị cho thấy có 296 đại lý thuộc nhóm "Nguy cơ" và "Giữ chân", mỗi nhóm chiếm khoảng 44% tổng số đại lý. Điều này cho thấy rằng gần một nửa số đại lý đang ở trong tình trạng cần được quan tâm và giữ chân.

Phân khúc RFM cho thấy có 170 đại lý thuộc nhóm "Champions" (25% tổng số đại lý), 141 đại lý thuộc nhóm "Loyal" (21% tổng số đại lý), và 112 đại lý thuộc nhóm "Lost" (17% tổng số đại lý). Điều này cho thấy rằng có một số đại lý trung thành và có tiềm năng, nhưng cũng có một số đại lý đã mất đi sự quan tâm và cần được phục hồi.

**2. Nhận định về nhóm đại lý nguy cơ rời bỏ**

Nhóm đại lý nguy cơ rời bỏ bao gồm 296 đại lý thuộc nhóm "Nguy cơ" và 43 đại lý thuộc nhóm "Cảnh báo". Top 10 đại lý nguy cơ cao nhất đều thuộc nhóm "Lost" và có xác suất đặt hàng thấp (< 0,05). Điều này cho thấy rằng những đại lý này đã mất đi sự quan tâm và không có kế hoạch đặt hàng 

In [7]:
exec_prompt = f"""Bạn là Giám đốc phân tích dữ liệu của Thống Nhất Bike. Hãy viết BÁO CÁO TÓM TẮT ĐIỀU HÀNH cho Ban Giám đốc về dự báo Q2/2026.\n\nTổng hợp từ 3 phân tích:\n\n1. DỰ BÁO DOANH SỐ:\n- Tổng doanh thu dự kiến Q2: {monthly['revenue'].sum():,.0f} VND\n- Tổng số lượng: {monthly['quantity'].sum():,.0f} xe\n\n2. DỰ BÁO MÀU SẮC:\n- Số SKU bán chậm: {slow_count.get('Bán chậm', 0)}\n- Số SKU giảm: {slow_count.get('Giảm', 0)}\n\n3. DỰ BÁO ĐẠI LÝ:\n- Tổng đại lý: {len(dealers)}\n- Đại lý nguy cơ: {priority_dist.get('Nguy cơ', 0)}\n- Đại lý cần cảnh báo: {priority_dist.get('Cảnh báo', 0)}\n\nHãy viết báo cáo ngắn gọn (khoảng 500 từ), bao gồm:\n1. Tổng quan tình hình\n2. Điểm nổi bật\n3. Rủi ro cần lưu ý\n4. Khuyến nghị hành động (3-5 điểm cụ thể)"""
exec_summary = ask_llm(exec_prompt, SYSTEM)
print('=== Executive Summary ===')
print(exec_summary)

=== Executive Summary ===
**BÁO CÁO TÓM TẮT ĐIỀU HÀNH Q2/2026**

**1. Tổng quan tình hình**

Trong quý 2 năm 2026, Thống Nhất Bike dự kiến đạt tổng doanh thu khoảng 26,421,507,842 VND, với tổng số lượng xe bán ra là 15,420 chiếc. Đây là một tín hiệu tích cực cho thấy sự ổn định và tăng trưởng của công ty trong thị trường xe đạp.

**2. Điểm nổi bật**

Một số điểm nổi bật trong quý 2 năm 2026 bao gồm:

* Tổng doanh thu dự kiến tăng so với cùng kỳ năm trước, cho thấy sự tăng trưởng ổn định của công ty.
* Số lượng xe bán ra cũng tăng, cho thấy nhu cầu về xe đạp của thị trường vẫn còn cao.
* Tuy nhiên, cũng có một số màu sắc và SKU bán chậm, cần được xem xét và điều chỉnh để tối ưu hóa sản xuất và kinh doanh.

**3. Rủi ro cần lưu ý**

Một số rủi ro cần lưu ý trong quý 2 năm 2026 bao gồm:

* Số lượng đại lý nguy cơ cao (296) và đại lý cần cảnh báo (43) có thể ảnh hưởng đến việc phân phối và bán hàng của công ty.
* Số lượng SKU bán chậm (11) và giảm (9) có thể dẫn đến tồn kho và lãng phí nếu 

In [8]:
insights = {
    'q1_analysis': q1_analysis,
    'q2_analysis': q2_analysis,
    'q3_analysis': q3_analysis,
    'executive_summary': exec_summary,
    'model': MODEL,
    'generated_at': pd.Timestamp.now().isoformat()
}
with open(FCAST / 'llm_insights.json', 'w', encoding='utf-8') as f:
    json.dump(insights, f, ensure_ascii=False, indent=2)
print(f'Saved llm_insights.json ({Path(FCAST / "llm_insights.json").stat().st_size/1024:.1f} KB)')

Saved llm_insights.json (10.4 KB)
